In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import csv
import time
import json

In [2]:
# Load the Dataset

In [3]:
text = ""
with open("/kaggle/input/datasets/ayush120/poems-dataset/poems-100.csv", "r") as file:
    reader = csv.reader(file)
    for row in reader:
        text += " ".join(row) + " "

In [4]:
# Tokenize the Text into Words

In [5]:
tokens = text.split()

In [6]:
# Create a Dictionary to Map Words to Indices

In [7]:
word_to_idx = {}
idx_to_word = {}
vocab_size = 0

for word in tokens:
    if word not in word_to_idx:
        word_to_idx[word] = vocab_size
        idx_to_word[vocab_size] = word
        vocab_size += 1

In [8]:
# Convert Tokens to Indices

In [9]:
token_indices = [word_to_idx[word] for word in tokens]

In [10]:
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 7460


In [11]:
# Create Sequences and Targets

In [12]:
seq_length = 10
sequences = []
targets = []

for i in range(len(token_indices) - seq_length):
    seq = token_indices[i:i + seq_length]
    target = token_indices[i + seq_length]
    sequences.append(seq)
    targets.append(target)

In [13]:
# Convert to PyTorch Tensors

In [14]:
sequences = torch.tensor(sequences, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)

In [15]:
# Define One-Hot Encoding for RNN Model

In [16]:
class OneHotRNN(nn.Module):
    def __init__(self, vocab_size, hidden_dim, output_dim):
        super(OneHotRNN, self).__init__()
        self.rnn = nn.RNN(vocab_size, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        output, _ = self.rnn(x)
        out = self.fc(output[:, -1, :])
        return out

In [17]:
# Define LSTM Model with Embedding Layer

In [18]:
class PoemLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(PoemLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        out = self.fc(output[:, -1, :])
        return out

In [19]:
# Hyperparameters

In [20]:
embed_dim = 100
hidden_dim = 128
output_dim = vocab_size

In [21]:
# Initialize Models

In [22]:
onehot_model = OneHotRNN(vocab_size, hidden_dim, output_dim)
embedding_model = PoemLSTM(vocab_size, embed_dim, hidden_dim, output_dim)

In [23]:
criterion = nn.CrossEntropyLoss()
onehot_optimizer = optim.Adam(onehot_model.parameters(), lr=0.001)
embedding_optimizer = optim.Adam(embedding_model.parameters(), lr=0.001)

In [24]:
# Loss Tracking

In [25]:
onehot_losses, embedding_losses = [], []

In [26]:
# T raining Function

In [27]:
def train_model(model, optimizer, name):
    start_time = time.time()
    for epoch in range(100):
        total_loss = 0
        for i in range(0, len(sequences), 32):
            batch_seq = sequences[i:i + 32]
            batch_target = targets[i:i + 32]

            # One-Hot Encoding for OneHotRNN
            if name == "OneHotRNN":
                batch_seq = F.one_hot(batch_seq, num_classes=vocab_size).float()

            # Forward Pass
            outputs = model(batch_seq)
            loss = criterion(outputs, batch_target)

            # Backward Pass and Optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / (len(sequences) // 32)
        if name == "OneHotRNN":
            onehot_losses.append(avg_loss)
        else:
            embedding_losses.append(avg_loss)

        print(f"{name} Epoch [{epoch+1}/100], Avg Loss: {avg_loss:.4f}")
    print(f"{name} Training Time: {time.time() - start_time:.2f}s\n")

In [28]:
# Train Both Models

In [29]:
train_model(onehot_model, onehot_optimizer, "OneHotRNN")
train_model(embedding_model, embedding_optimizer, "EmbeddingLSTM")

OneHotRNN Epoch [1/100], Avg Loss: 7.5808
OneHotRNN Epoch [2/100], Avg Loss: 6.6927
OneHotRNN Epoch [3/100], Avg Loss: 6.3786
OneHotRNN Epoch [4/100], Avg Loss: 6.1341
OneHotRNN Epoch [5/100], Avg Loss: 5.9734
OneHotRNN Epoch [6/100], Avg Loss: 5.8493
OneHotRNN Epoch [7/100], Avg Loss: 5.6323
OneHotRNN Epoch [8/100], Avg Loss: 5.4560
OneHotRNN Epoch [9/100], Avg Loss: 5.2903
OneHotRNN Epoch [10/100], Avg Loss: 5.1189
OneHotRNN Epoch [11/100], Avg Loss: 4.8579
OneHotRNN Epoch [12/100], Avg Loss: 4.6013
OneHotRNN Epoch [13/100], Avg Loss: 4.3586
OneHotRNN Epoch [14/100], Avg Loss: 4.1171
OneHotRNN Epoch [15/100], Avg Loss: 3.7683
OneHotRNN Epoch [16/100], Avg Loss: 3.3953
OneHotRNN Epoch [17/100], Avg Loss: 3.0848
OneHotRNN Epoch [18/100], Avg Loss: 2.8988
OneHotRNN Epoch [19/100], Avg Loss: 2.6756
OneHotRNN Epoch [20/100], Avg Loss: 2.3498
OneHotRNN Epoch [21/100], Avg Loss: 2.0412
OneHotRNN Epoch [22/100], Avg Loss: 1.7707
OneHotRNN Epoch [23/100], Avg Loss: 1.5811
OneHotRNN Epoch [24/

In [30]:
# Poem Generation Function (Testing)

In [31]:
def generate_poem(model, seed_text, num_words=50, model_type="EmbeddingLSTM"):
    model.eval()
    words = seed_text.split()
    with torch.no_grad():
        for _ in range(num_words):
            seq = [word_to_idx.get(word, 0) for word in words[-seq_length:]]
            seq = torch.tensor(seq, dtype=torch.long).unsqueeze(0)

            if model_type == "OneHotRNN":
                seq = F.one_hot(seq, num_classes=vocab_size).float()

            output = model(seq)
            probabilities = F.softmax(output, dim=1)
            predicted_idx = torch.multinomial(probabilities, 1).item()

            words.append(idx_to_word[predicted_idx])

    return " ".join(words)

In [32]:
# Test Generation

In [33]:
seed_text = "I wandered lonely as a"
print("\nGenerated Poem (OneHotRNN):", generate_poem(onehot_model, seed_text, model_type="OneHotRNN"))
print("\nGenerated Poem (EmbeddingLSTM):", generate_poem(embedding_model, seed_text, model_type="EmbeddingLSTM"))


Generated Poem (OneHotRNN): I wandered lonely as a These place up how days may become to me, if I could tell that is in them who and air. shall but as much as myself, at last The latter I translate and let us shuddering to my spirit When I lie down to be headland and past and place

Generated Poem (EmbeddingLSTM): I wandered lonely as a while, Sweet my own crucifixion and bloody crowning. I remember now, I resume the overstaid fraction, The grave of rock multiplies what has been confided to it, or to any graves, Corpses rise, gashes heal, fastenings roll from me. I troop forth replenish'd with supreme power, one of an average


In [34]:
# Exporting Models

In [35]:
print("\n" + "="*60)
print("EXPORTING MODELS")
print("="*60)


EXPORTING MODELS


In [36]:
# Save OneHotRNN

In [37]:
torch.save({
    'model_state_dict': onehot_model.state_dict(),
    'vocab_size': vocab_size,
    'hidden_dim': hidden_dim,
}, 'onehot_rnn.pt')
print("✅ Saved onehot_rnn.pt")

✅ Saved onehot_rnn.pt


In [38]:
# Save EmbeddingLSTM

In [39]:
torch.save({
    'model_state_dict': embedding_model.state_dict(),
    'vocab_size': vocab_size,
    'embed_dim': embed_dim,
    'hidden_dim': hidden_dim,
}, 'embedding_lstm.pt')
print("✅ Saved embedding_lstm.pt")

✅ Saved embedding_lstm.pt


In [40]:
# Save Vocabulary

In [41]:
vocab_data = {
    'word_to_idx': word_to_idx,
    'idx_to_word': {str(k): v for k, v in idx_to_word.items()},
}
with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f)
print("✅ Saved vocab.json")

✅ Saved vocab.json
